# Build the dataset

Raw CSVs -> features ready for training, saved to `OUTPUT_DIR` (`../data/experiment_X/`).

1. **Normalization** (`normalize.py`): special codes and units -> plain numbers, column by column.
2. **Preprocessing** (`preprocessing.py`): drop identifier / survey design, constant and incomplete
   columns; add missing indicators; drop one of each highly correlated feature pair, then columns
   weakly correlated with the output; one-hot expand category columns; optional leak features; clip outliers, fill NaN,
   standardize, add polynomial columns, add bias. Statistics are fitted on `x_train` only and reused for `x_test`.
3. **Labels**: {-1, 1} for least squares / ridge, {0, 1} for logistic regression.

## 0. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
sys.path.append("..")  # so we can import the modules from the repo root

import numpy as np

from helpers import load_csv_data, load_csv_data_labels
from normalize import normalize_codes, normalize_nan
from preprocessing import (
    IDENTIFIER_COLUMNS,
    SURVEY_DESIGN_COLUMNS,
    CATEGORY_COLUMNS,
    drop_labels,
    drop_columns_by_name,
    find_constant_columns,
    filter_incomplete_column,
    add_missing_indicator_columns,
    find_correlated_columns,
    find_low_output_correlation_columns,
    expand_category_column,
    filter_outliers,
    add_polynomial_columns,
    compute_mean_std,
    standardize,
    add_bias,
)

DATA_DIR = "../data"
# Experiments:
#   experiment_1: no category expansion, QSTVER kept, INCOMPLETE_NAN_RATIO = 0.9, no missing indicators
#   experiment_2: + category expansion, QSTVER dropped
#   experiment_3: + INCOMPLETE_NAN_RATIO = 0.99, missing indicators
#   experiment_4: + squares of POLY_COLUMNS (built with a second standardization of every column, since fixed)
#   experiment_5: experiment_3 rebuilt by this notebook (POLY_COLUMNS = [], LEAK_COLUMNS = [])
#   experiment_6: experiment_5 + LEAK_COLUMNS = ["HAREHAB1"]
#   experiment_7: experiment_6 with LEAK_ONLY = True (bias + leak feature only)
# See README.md for the results of each experiment.
OUTPUT_DIR = "../data/experiment_6"
INCOMPLETE_NAN_RATIO = 0.99  # columns with more NaN are dropped
MISSING_INDICATOR_NAN_RATIO = 0.05  # columns with more NaN get a `column_missing` indicator (None: no indicators)
# Continuous columns whose log-odds per decile are curved (age, poor physical health days, alcohol, exercise)
POLY_COLUMNS = []  # experiment_4: ["_AGEG5YR", "PHYSHLTH", "DROCDY3_", "EXERHMM1"]
POLY_DEGREE = 2
# Label leak: HAREHAB1 ("rehab after your heart attack?") is only asked to people who had a heart attack,
# so every respondent who answered it is positive. Adds a 0/1 `column_answered` feature from the raw data.
LEAK_COLUMNS = ["HAREHAB1"]  # []: no leak feature
LEAK_ONLY = False  # True: keep only the bias and the leak features
SUB_SAMPLE = False  # True for quick iterations, False for the real dataset

np.random.seed(1)

## 1. Load data

In [ ]:
x_labels = load_csv_data_labels(DATA_DIR)
x_train, x_test, y_train, train_ids, test_ids = load_csv_data(DATA_DIR, sub_sample=SUB_SAMPLE)

print("x_train:", x_train.shape)
print("x_test: ", x_test.shape)
print("y_train:", y_train.shape, "unique labels:", np.unique(y_train))

## 2. Normalization

Codes first (don't know / refused -> NaN, none / never -> 0), then units (kg, cm, per day, ...).

In [ ]:
x_train_normalized = normalize_codes(x_labels, x_train)
x_test_normalized = normalize_codes(x_labels, x_test)

print("NaN ratio x_train: raw", np.isnan(x_train).mean(), "-> normalized", np.isnan(x_train_normalized).mean())

## 3. Preprocessing

### Drop identifier and survey design columns

In [ ]:
non_feature_columns = IDENTIFIER_COLUMNS + SURVEY_DESIGN_COLUMNS

labels = drop_labels(x_labels, non_feature_columns)
x_train_preprocessed = drop_columns_by_name(x_labels, x_train_normalized, non_feature_columns)
x_test_preprocessed = drop_columns_by_name(x_labels, x_test_normalized, non_feature_columns)

print("x_train_preprocessed:", x_train_preprocessed.shape)

### Drop constant columns (fitted on x_train)

In [ ]:
constant_columns = find_constant_columns(labels, x_train_preprocessed)
print(len(constant_columns), "constant columns:", constant_columns)

x_train_preprocessed = drop_columns_by_name(labels, x_train_preprocessed, constant_columns)
x_test_preprocessed = drop_columns_by_name(labels, x_test_preprocessed, constant_columns)
labels = drop_labels(labels, constant_columns)
print("x_train_preprocessed:", x_train_preprocessed.shape)

### Drop incomplete columns (fitted on x_train)

NaN ratio > `INCOMPLETE_NAN_RATIO`.

In [ ]:
incomplete_columns = filter_incomplete_column(labels, x_train_preprocessed, INCOMPLETE_NAN_RATIO)
print(len(incomplete_columns), "incomplete columns:", incomplete_columns)

x_train_preprocessed = drop_columns_by_name(labels, x_train_preprocessed, incomplete_columns)
x_test_preprocessed = drop_columns_by_name(labels, x_test_preprocessed, incomplete_columns)
labels = drop_labels(labels, incomplete_columns)
print("x_train_preprocessed:", x_train_preprocessed.shape)

### Add missing indicators (fitted on x_train)

Columns with a NaN ratio > `MISSING_INDICATOR_NAN_RATIO` get a 0/1 `column_missing` column, before NaN are
filled. Added before the correlation filters: indicators of questions from the same module are identical,
the feature - feature filter keeps one; the feature - output filter drops the uninformative ones.

In [ ]:
if MISSING_INDICATOR_NAN_RATIO is None:
    indicator_columns = []
else:
    indicator_columns = filter_incomplete_column(labels, x_train_preprocessed, MISSING_INDICATOR_NAN_RATIO)
print(len(indicator_columns), "missing indicators added")

_, x_test_preprocessed = add_missing_indicator_columns(labels, x_test_preprocessed, indicator_columns)
labels, x_train_preprocessed = add_missing_indicator_columns(labels, x_train_preprocessed, indicator_columns)
print("x_train_preprocessed:", x_train_preprocessed.shape)

### Drop correlated feature pairs (fitted on x_train)

Per pair with |correlation| >= `CORRELATION_THRESHOLD`, the one less correlated with the output.

In [ ]:
correlated_columns = find_correlated_columns(labels, x_train_preprocessed, y_train)
print(len(correlated_columns), "correlated columns:", correlated_columns)

x_train_preprocessed = drop_columns_by_name(labels, x_train_preprocessed, correlated_columns)
x_test_preprocessed = drop_columns_by_name(labels, x_test_preprocessed, correlated_columns)
labels = drop_labels(labels, correlated_columns)
print("x_train_preprocessed:", x_train_preprocessed.shape)

### Drop columns weakly correlated with the output (fitted on x_train)

|correlation| with y < `OUTPUT_CORRELATION_THRESHOLD`.

In [ ]:
low_output_correlation_columns = find_low_output_correlation_columns(labels, x_train_preprocessed, y_train)
print(len(low_output_correlation_columns), "low output correlation columns:", low_output_correlation_columns)

x_train_filtered = drop_columns_by_name(labels, x_train_preprocessed, low_output_correlation_columns)
x_test_filtered = drop_columns_by_name(labels, x_test_preprocessed, low_output_correlation_columns)
labels = drop_labels(labels, low_output_correlation_columns)
print("x_train_filtered:", x_train_filtered.shape)

### Expand category columns (categories from x_train)

Each column of `CATEGORY_COLUMNS` still present -> one 0/1 column `column_X` per category X seen in x_train.

In [ ]:
expanded_columns = [name for name in CATEGORY_COLUMNS if name in labels.values()]
print("not present (filtered out):", [name for name in CATEGORY_COLUMNS if name not in expanded_columns])

for name in expanded_columns:
    train_labels, x_train_filtered, categories = expand_category_column(labels, x_train_filtered, name)
    _, x_test_filtered, _ = expand_category_column(labels, x_test_filtered, name, categories)
    labels = train_labels

print(len(expanded_columns), "columns expanded")
print("x_train_filtered:", x_train_filtered.shape, "x_test_filtered:", x_test_filtered.shape)

### Add leak features (optional, see `LEAK_COLUMNS`)

1 if the question was answered in the **raw** data (before code normalization, so "don't know / refused"
answers count as answered), else 0.

In [ ]:
raw_index = {name: i for (i, name) in x_labels.items()}
leak_indices = [raw_index[name] for name in LEAK_COLUMNS]

x_train_filtered = np.c_[x_train_filtered, ~np.isnan(x_train[:, leak_indices])]
x_test_filtered = np.c_[x_test_filtered, ~np.isnan(x_test[:, leak_indices])]
labels = dict(enumerate(list(labels.values()) + [f"{name}_answered" for name in LEAK_COLUMNS]))

print(len(LEAK_COLUMNS), "leak features added:", LEAK_COLUMNS)
print("x_train_filtered:", x_train_filtered.shape, "x_test_filtered:", x_test_filtered.shape)

### Clip outliers (train bounds reused for x_test)

Continuous columns clipped to their [`OUTLIER_RATIO`, 1 - `OUTLIER_RATIO`] quantiles.

In [ ]:
x_train_filtered, outlier_bounds = filter_outliers(x_train_filtered)
x_test_filtered, _ = filter_outliers(x_test_filtered, bounds=outlier_bounds)

clipped_columns = np.flatnonzero(np.isfinite(outlier_bounds[0]))
print(len(clipped_columns), "continuous columns clipped")

### Fill NaN, standardize, add polynomial columns, add bias (train statistics reused for x_test)

Powers are taken on the standardized (centered) values, then standardized themselves.

In [ ]:
train_mean, train_std = compute_mean_std(x_train_filtered)
x_train_standardized = standardize(normalize_nan(x_train_filtered, train_mean), train_mean, train_std)
x_test_standardized = standardize(normalize_nan(x_test_filtered, train_mean), train_mean, train_std)

poly_columns = [name for name in POLY_COLUMNS if name in labels.values()]
print("not present (filtered out):", [name for name in POLY_COLUMNS if name not in poly_columns])
n_columns = x_train_standardized.shape[1]
_, x_test_standardized = add_polynomial_columns(labels, x_test_standardized, poly_columns, POLY_DEGREE)
labels, x_train_standardized = add_polynomial_columns(labels, x_train_standardized, poly_columns, POLY_DEGREE)

# Standardize the new power columns only
poly_mean, poly_std = compute_mean_std(x_train_standardized[:, n_columns:])
x_train_standardized[:, n_columns:] = standardize(x_train_standardized[:, n_columns:], poly_mean, poly_std)
x_test_standardized[:, n_columns:] = standardize(x_test_standardized[:, n_columns:], poly_mean, poly_std)

tx_train = add_bias(x_train_standardized)
tx_test = add_bias(x_test_standardized)

feature_names = np.array(["bias"] + list(labels.values()))

print("tx_train:", tx_train.shape, "tx_test:", tx_test.shape)
assert not np.isnan(tx_train).any() and not np.isnan(tx_test).any()

### Leak only (optional, see `LEAK_ONLY`)

Keeps only the bias and the leak features: how much the leak alone is worth.

In [ ]:
if LEAK_ONLY:
    kept = [0] + [j for j, name in enumerate(feature_names) if name.endswith("_answered")]
    tx_train, tx_test, feature_names = tx_train[:, kept], tx_test[:, kept], feature_names[kept]

print("features:", list(feature_names) if LEAK_ONLY else len(feature_names))
print("tx_train:", tx_train.shape, "tx_test:", tx_test.shape)

## 4. Labels

`y_train` in {-1, 1} (least squares, ridge, submission format), `y_train_01` in {0, 1} (logistic regression).

In [ ]:
y_train_01 = (y_train == 1).astype(float)

print("P(y = 1):", y_train_01.mean())

## 5. Save

`x_train.npy`, `x_test.npy`, `y_train.npy` in `OUTPUT_DIR`.

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

np.save(os.path.join(OUTPUT_DIR, "x_train.npy"), tx_train)
np.save(os.path.join(OUTPUT_DIR, "x_test.npy"), tx_test)
np.save(os.path.join(OUTPUT_DIR, "y_train.npy"), y_train)
print("saved to", OUTPUT_DIR)

In [ ]:
# Reload check
for name in ["x_train", "x_test", "y_train"]:
    print(name, np.load(os.path.join(OUTPUT_DIR, f"{name}.npy")).shape)